## Import Libraries and Datasets

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

DATA_DIR = Path("data")
sorted(p.name for p in DATA_DIR.iterdir())

['bars_1m.parquet',
 'events.json',
 'exchange_calendar.csv',
 'regime_labels.csv',
 'sim_runs.json',
 'stream_bars.json',
 'symbol_history.csv',
 'symbols.csv',
 'vendor_daily.csv']

In [2]:
exchange_calendar = pd.read_csv(DATA_DIR / "exchange_calendar.csv")
symbol_history = pd.read_csv(DATA_DIR / "symbol_history.csv")
symbols = pd.read_csv(DATA_DIR / "symbols.csv")
vendor_daily = pd.read_csv(DATA_DIR / "vendor_daily.csv")


bars = pd.read_parquet(DATA_DIR / "bars_1m.parquet")


for name, df in [
    ("exchange_calendar", exchange_calendar),
    ("symbol_history", symbol_history),
    ("symbols", symbols),
    ("vendor_daily", vendor_daily),
    ("bars", bars),
]:
    print(f"{name:16s} {df.shape}  {list(df.columns)[:6]} ...")

exchange_calendar (129, 6)  ['date', 'is_trading_day', 'session_open', 'session_close', 'timezone', 'note'] ...
symbol_history   (3, 5)  ['security_id', 'change_type', 'old_value', 'new_value', 'effective_date'] ...
symbols          (31, 7)  ['security_id', 'ticker', 'company_name', 'sector', 'listing_date', 'delisting_date'] ...
vendor_daily     (10831, 8)  ['run_id', 'date', 'ticker', 'open', 'high', 'low'] ...
bars             (4276175, 9)  ['run_id', 'ticker', 'ts', 'open', 'high', 'low'] ...


## Flatten nested JSON

In [4]:
import json

with open(DATA_DIR / "sim_runs.json", encoding="utf-8") as f:
    sim_runs_raw = json.load(f)

with open(DATA_DIR / "events.json", encoding="utf-8") as f:
    events_raw = json.load(f)

In [5]:
sim_runs_flat = pd.json_normalize(sim_runs_raw, sep="_")
sim_runs_flat.insert(0, "source_row", range(1, len(sim_runs_flat) + 1))

events_flat = pd.json_normalize(events_raw["events"], sep="_")
events_flat.insert(0, "source_row", range(1, len(events_flat) + 1))

In [ ]:
print("SIM_RUNS")
print(sim_runs_flat.shape)
print(sim_runs_flat.columns.tolist())
display(sim_runs_flat.head())

print("\nEVENTS")
print(events_flat.shape)
print(events_flat.columns.tolist())
display(events_flat.head())

SIM_RUNS
(4, 25)
['source_row', 'run_id', 'config_version', 'created_at', 'seed', 'window_start', 'window_end', 'universe_stocks', 'universe_index_ticker', 'universe_index_method', 'universe_index_base_level', 'parameters_volatility_sigma_annual', 'parameters_climate_states', 'parameters_climate_variance_multiplier', 'parameters_climate_transition_matrix', 'parameters_news_memory_alpha', 'parameters_news_memory_beta', 'parameters_news_memory_gamma', 'parameters_intraday_open_boost', 'parameters_intraday_close_boost', 'parameters_shocks_distribution', 'parameters_shocks_df', 'parameters_jumps_rate_per_day', 'parameters_factors_market_weight', 'parameters_factors_sector_share']


,source_row,run_id,config_version,created_at,seed,window_start,window_end,universe_stocks,universe_index_ticker,universe_index_method,...,parameters_news_memory_alpha,parameters_news_memory_beta,parameters_news_memory_gamma,parameters_intraday_open_boost,parameters_intraday_close_boost,parameters_shocks_distribution,parameters_shocks_df,parameters_jumps_rate_per_day,parameters_factors_market_weight,parameters_factors_sector_share
0,1,RUN-2025A,1.4.0,2025-07-02T08:00:00+02:00,8522640,2025-01-02,2025-06-30,30,INDEX,cap_weighted,...,0.03,0.8,0.6,1.2,0.6,student_t,4,0.02,"[0.12, 0.22, 0.45]",0.25
1,2,RUN-2025B,1.4.1,2025-07-02T08:00:00+02:00,7737823,2025-01-02,2025-06-30,30,INDEX,cap_weighted,...,0.33,0.8,0.0,0.0,0.0,student_t,4,0.02,"[0.12, 0.22, 0.45]",0.25
2,3,RUN-2025C,1.4.0,2025-07-02T08:00:00+02:00,8325121,2025-01-02,2025-06-30,30,INDEX,cap_weighted,...,0.03,0.8,0.6,1.2,0.6,student_t,4,0.02,"[0.12, 0.22, 0.45]",0.25
3,4,RUN-2025B,1.4.1,2025-07-03T11:20:00+02:00,7737823,2025-01-02,2025-06-30,30,INDEX,cap_weighted,...,0.33,0.8,0.0,0.0,0.0,student_t,4,0.02,"[0.12, 0.22, 0.45]",0.25



EVENTS
(62, 8)
['source_row', 'event_id', 'type', 'time', 'security_ticker', 'details_session', 'details_duration_minutes', 'details_ratio']


,source_row,event_id,type,time,security_ticker,details_session,details_duration_minutes,details_ratio
0,1,EV0001,earnings,2025-01-10T09:30:00-05:00,DALN,pre_open,NaN,NaN
1,2,EV0002,earnings,2025-01-14T09:30:00-05:00,BRIS,pre_open,NaN,NaN
2,3,EV0003,earnings,2025-01-15T09:30:00-05:00,SOLK,pre_open,NaN,NaN
3,4,EV0004,earnings,2025-01-24T09:30:00-05:00,XERN,pre_open,NaN,NaN
4,5,EV0005,earnings,2025-01-30T09:30:00-05:00,VALX,pre_open,NaN,NaN


In [7]:
bars.head()

,run_id,ticker,ts,open,high,low,close,volume,received_at
0,RUN-2025A,ARVT,2025-01-02 09:30:00,120.07,120.08,119.84,119.93,8570,2025-01-02 14:31:00+00:00
1,RUN-2025A,ARVT,2025-01-02 09:31:00,119.93,119.98,119.35,119.39,45001,2025-01-02 14:32:01+00:00
2,RUN-2025A,ARVT,2025-01-02 09:32:00,119.39,119.66,119.30,119.53,5825,2025-01-02 14:33:00+00:00
3,RUN-2025A,ARVT,2025-01-02 09:33:00,119.53,119.57,119.23,119.35,8195,2025-01-02 14:34:00+00:00
4,RUN-2025A,ARVT,2025-01-02 09:34:00,119.35,119.44,119.33,119.38,5631,2025-01-02 14:35:01+00:00


In [8]:
sim_runs_flat.head()

,source_row,run_id,config_version,created_at,seed,window_start,window_end,universe_stocks,universe_index_ticker,universe_index_method,...,parameters_news_memory_alpha,parameters_news_memory_beta,parameters_news_memory_gamma,parameters_intraday_open_boost,parameters_intraday_close_boost,parameters_shocks_distribution,parameters_shocks_df,parameters_jumps_rate_per_day,parameters_factors_market_weight,parameters_factors_sector_share
0,1,RUN-2025A,1.4.0,2025-07-02T08:00:00+02:00,8522640,2025-01-02,2025-06-30,30,INDEX,cap_weighted,...,0.03,0.8,0.6,1.2,0.6,student_t,4,0.02,"[0.12, 0.22, 0.45]",0.25
1,2,RUN-2025B,1.4.1,2025-07-02T08:00:00+02:00,7737823,2025-01-02,2025-06-30,30,INDEX,cap_weighted,...,0.33,0.8,0.0,0.0,0.0,student_t,4,0.02,"[0.12, 0.22, 0.45]",0.25
2,3,RUN-2025C,1.4.0,2025-07-02T08:00:00+02:00,8325121,2025-01-02,2025-06-30,30,INDEX,cap_weighted,...,0.03,0.8,0.6,1.2,0.6,student_t,4,0.02,"[0.12, 0.22, 0.45]",0.25
3,4,RUN-2025B,1.4.1,2025-07-03T11:20:00+02:00,7737823,2025-01-02,2025-06-30,30,INDEX,cap_weighted,...,0.33,0.8,0.0,0.0,0.0,student_t,4,0.02,"[0.12, 0.22, 0.45]",0.25


RUN-2025B appears twice in sim_runs; same config_version, but created_at differs. 
For the bar join, should retain one run record and exclude conflicting "created_at" as the relation is many_to_one

In [10]:
sim_runs_clean = (
    sim_runs_flat[["run_id", "config_version"]]
    .drop_duplicates(subset=["run_id", "config_version"])
)

In [11]:
bars_with_runs= bars.merge(
    sim_runs_clean,
    on="run_id",
    how="left",
    validate="many_to_one"
)

In [12]:
print("bars before:", len(bars))
print("bars after: ", len(bars_with_runs))
print("unmatched bars:", bars_with_runs["config_version"].isna().sum())

bars before: 4276175
bars after:  4276175
unmatched bars: 260


In [13]:
bars_with_runs.loc[
    bars_with_runs["config_version"].isna(),
    "run_id"
].drop_duplicates()

2905314    RUN-TEST
Name: run_id, dtype: str

### Bars with an unknown run

**Finding:** The bars table contains 260 rows with `run_id = RUN-TEST`. This ID does not appear in the flattened `sim_runs` table.

**How it was detected:** I compared the distinct `run_id` values in `bars` with those in `sim_runs_flat`. The `RUN-TEST` bars did not match a known simulation run.

**Decision:** These bars are candidates for rejection because their run cannot be verified. For each rejected bar, the rejects table preserves the raw bar payload, source file (`bars_1m.parquet`), source row, and one rejection reason: `UNKNOWN_RUN`.

In [14]:
bars = bars.copy()
bars["_source_row"] = range(1, len(bars) + 1)

In [15]:
unknown_run_bars = bars.loc[bars["run_id"] == "RUN-TEST"]

reject_rows = [
    {
        "raw_payload": row.drop(labels="_source_row").to_dict(),
        "source_file": "bars_1m.parquet",
        "source_row": int(row["_source_row"]),
        "reason": "UNKNOWN_RUN",
    }
    for _, row in unknown_run_bars.iterrows()
]

rejects = pd.DataFrame(reject_rows)

display(rejects)

,raw_payload,source_file,source_row,reason
0,"{'run_id': 'RUN-TEST', 'ticker': 'BELX', 'ts':...",bars_1m.parquet,2905315,UNKNOWN_RUN
1,"{'run_id': 'RUN-TEST', 'ticker': 'BELX', 'ts':...",bars_1m.parquet,2905319,UNKNOWN_RUN
2,"{'run_id': 'RUN-TEST', 'ticker': 'BELX', 'ts':...",bars_1m.parquet,2905323,UNKNOWN_RUN
3,"{'run_id': 'RUN-TEST', 'ticker': 'BELX', 'ts':...",bars_1m.parquet,2905327,UNKNOWN_RUN
4,"{'run_id': 'RUN-TEST', 'ticker': 'BELX', 'ts':...",bars_1m.parquet,2905331,UNKNOWN_RUN
...,...,...,...,...
255,"{'run_id': 'RUN-TEST', 'ticker': 'BELX', 'ts':...",bars_1m.parquet,2906335,UNKNOWN_RUN
256,"{'run_id': 'RUN-TEST', 'ticker': 'BELX', 'ts':...",bars_1m.parquet,2906339,UNKNOWN_RUN
257,"{'run_id': 'RUN-TEST', 'ticker': 'BELX', 'ts':...",bars_1m.parquet,2906343,UNKNOWN_RUN
258,"{'run_id': 'RUN-TEST', 'ticker': 'BELX', 'ts':...",bars_1m.parquet,2906347,UNKNOWN_RUN
